# Notebook 02 — Image Preprocessing Pipeline

**EvaliSense: AI-Powered Handwritten Examination Evaluation with ML-Based Grading Error Prediction**

---

## Objective

This notebook demonstrates the image preprocessing pipeline that prepares handwritten answer sheets for recognition.

We will:
1. Load the `ImagePreprocessor` module and inspect its configuration
2. Walk through each preprocessing step (grayscale → blur → binarize → clean → deskew)
3. Visualize before/after on generated test answer sheets
4. Apply preprocessing to real IAM handwriting images
5. Measure preprocessing time on different image sizes
6. Compare binarization techniques (Otsu vs Adaptive)

**Why this matters:** Raw scanned images contain noise, skew, and uneven lighting that degrade OCR accuracy. Proper preprocessing is essential before feeding images to TrOCR.

## Setup

In [ ]:
import os, sys, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import config
from preprocessing import ImagePreprocessor

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 120

RESULTS_DIR = config.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

preprocessor = ImagePreprocessor()

print(f"Project root: {PROJECT_ROOT}")
print(f"ImagePreprocessor: {type(preprocessor).__name__}")
print(f"OpenCV version: {cv2.__version__}")
print("Setup complete ✓")

## 1. Full Preprocessing Pipeline

The `ImagePreprocessor` module applies a sequence of transformations:

1. **Load** — Read the image from disk
2. **Grayscale** — Convert BGR to single-channel
3. **Noise Reduction** — Gaussian blur to smooth noise
4. **Binarization** — Convert to binary (black text on white background)
5. **Morphological Cleaning** — Remove small artifacts
6. **Deskewing** — Correct rotated scans

In [ ]:
# Find a test image
test_images = sorted(config.raw_dir.glob('answer_*.png'))
if test_images:
    test_path = test_images[0]
    print(f"Test image: {test_path.name}")
    
    original = cv2.imread(str(test_path))
    print(f"Original shape: {original.shape}")
    print(f"Original size: {original.nbytes / 1024 / 1024:.1f} MB")
    
    # Run full preprocessing
    t0 = time.time()
    result = preprocessor.process(str(test_path))
    preprocess_time = time.time() - t0
    
    print(f"\nProcessed shape: {result.final.shape}")
    print(f"Processing time: {preprocess_time:.3f}s")
    print(f"Metadata: {result.metadata}")
else:
    print("No test images found. Run: python create_test_images.py")

In [ ]:
# Before/After comparison
if test_images:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 12))
    fig.suptitle('Image Preprocessing: Before vs After', fontsize=16, fontweight='bold')
    
    crop_h = min(1500, original.shape[0])
    
    # Original
    ax1.imshow(cv2.cvtColor(original[:crop_h], cv2.COLOR_BGR2RGB))
    ax1.set_title(f'Original ({original.shape[1]}×{original.shape[0]})', fontsize=13)
    ax1.axis('off')
    
    # Preprocessed
    processed = result.final
    if processed.ndim == 2:
        ax2.imshow(processed[:crop_h], cmap='gray')
    else:
        ax2.imshow(cv2.cvtColor(processed[:crop_h], cv2.COLOR_BGR2RGB))
    ax2.set_title(f'Preprocessed ({processed.shape[1]}×{processed.shape[0]})', fontsize=13)
    ax2.axis('off')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'preprocessing_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {RESULTS_DIR / 'preprocessing_comparison.png'}")

## 2. Step-by-Step Breakdown

Let's visualize each individual preprocessing step to understand its effect.

In [ ]:
if test_images:
    img = cv2.imread(str(test_path))
    crop_h = 1200
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 14))
    fig.suptitle('Preprocessing Pipeline — Step-by-Step Breakdown', fontsize=16, fontweight='bold')
    
    # Step 1: Original
    axes[0, 0].imshow(cv2.cvtColor(img[:crop_h], cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title('Step 1: Original Image', fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Step 2: Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    axes[0, 1].imshow(gray[:crop_h], cmap='gray')
    axes[0, 1].set_title('Step 2: Grayscale Conversion', fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')
    
    # Step 3: Gaussian Blur
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    axes[0, 2].imshow(blurred[:crop_h], cmap='gray')
    axes[0, 2].set_title('Step 3: Gaussian Blur (Noise Reduction)', fontsize=12, fontweight='bold')
    axes[0, 2].axis('off')
    
    # Step 4: Otsu Binarization
    _, otsu = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    axes[1, 0].imshow(otsu[:crop_h], cmap='gray')
    axes[1, 0].set_title('Step 4: Otsu Binarization', fontsize=12, fontweight='bold')
    axes[1, 0].axis('off')
    
    # Step 5: Adaptive Threshold (alternative)
    adaptive = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 15, 10)
    axes[1, 1].imshow(adaptive[:crop_h], cmap='gray')
    axes[1, 1].set_title('Step 5: Adaptive Gaussian Threshold', fontsize=12, fontweight='bold')
    axes[1, 1].axis('off')
    
    # Step 6: Morphological Cleaning
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    cleaned = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, kernel)
    axes[1, 2].imshow(cleaned[:crop_h], cmap='gray')
    axes[1, 2].set_title('Step 6: Morphological Cleaning', fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'preprocessing_steps.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {RESULTS_DIR / 'preprocessing_steps.png'}")

## 3. Apply Preprocessing to IAM Images

Let's apply the same pipeline to real handwriting from the IAM dataset to see how it handles natural handwriting.

In [ ]:
iam_test_dir = config.data_dir / 'external' / 'iam_lines' / 'test'
iam_images = sorted(iam_test_dir.glob('*.png'))[:6] if iam_test_dir.exists() else []

if iam_images:
    fig, axes = plt.subplots(len(iam_images), 2, figsize=(16, 3 * len(iam_images)))
    fig.suptitle('IAM Images — Original vs Preprocessed', fontsize=16, fontweight='bold', y=1.01)
    
    for idx, img_path in enumerate(iam_images):
        img = cv2.imread(str(img_path))
        
        # Original
        axes[idx, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[idx, 0].set_title(f'Original: {img_path.name}', fontsize=9)
        axes[idx, 0].axis('off')
        
        # Preprocessed
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        axes[idx, 1].imshow(binary, cmap='gray')
        axes[idx, 1].set_title('Preprocessed (Otsu)', fontsize=9)
        axes[idx, 1].axis('off')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'iam_preprocessing.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {RESULTS_DIR / 'iam_preprocessing.png'}")
else:
    print("No IAM images found.")

## 4. Binarization Technique Comparison

Compare Otsu thresholding vs Adaptive Gaussian thresholding on the same image.

In [ ]:
if iam_images:
    sample_img = cv2.imread(str(iam_images[0]), cv2.IMREAD_GRAYSCALE)
    blurred = cv2.GaussianBlur(sample_img, (3, 3), 0)
    
    # Three techniques
    _, otsu = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptive_mean = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                           cv2.THRESH_BINARY, 15, 10)
    adaptive_gauss = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                            cv2.THRESH_BINARY, 15, 10)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle('Binarization Technique Comparison', fontsize=14, fontweight='bold')
    
    for ax, img, title in zip(axes, [sample_img, otsu, adaptive_mean, adaptive_gauss],
                               ['Original (Grayscale)', 'Otsu Thresholding', 
                                'Adaptive Mean', 'Adaptive Gaussian']):
        ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'binarization_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {RESULTS_DIR / 'binarization_comparison.png'}")

## 5. Preprocessing Timing

Measure how long preprocessing takes on images of different sizes.

In [ ]:
# Benchmark preprocessing time on available images
all_test_images = list(config.raw_dir.glob('answer_*.png'))[:4]

if all_test_images:
    print(f"{'Image':<45} {'Shape':>20} {'Time (ms)':>12}")
    print("─" * 80)
    
    times = []
    for img_path in all_test_images:
        img = cv2.imread(str(img_path))
        t0 = time.time()
        preprocessor.process(str(img_path))
        elapsed = (time.time() - t0) * 1000
        times.append(elapsed)
        print(f"  {img_path.name:<43} {str(img.shape):>20} {elapsed:>10.1f}ms")
    
    print(f"\n  Average: {np.mean(times):.1f}ms | Max: {max(times):.1f}ms | Min: {min(times):.1f}ms")
    print(f"  Device: CPU (Intel Core Ultra 9 285H)")

## Observations

1. **Otsu vs Adaptive:** Otsu works well for evenly-lit images. Adaptive Gaussian handles uneven lighting better — important for real exam scans.

2. **Morphological Cleaning:** The MORPH_CLOSE operation fills small gaps in strokes, improving recognition for thin or broken handwriting.

3. **Speed:** Preprocessing runs in <500ms per image on CPU, making it suitable for real-time evaluation.

4. **IAM Images:** IAM line images are already fairly clean. The preprocessing primarily helps with the generated test images that simulate real exam conditions.

## Conclusion

The preprocessing pipeline is CPU-efficient and handles both clean (IAM) and noisy (scanned) images. Otsu binarization is used as the default, with adaptive thresholding as a fallback.

**Next:** [Notebook 03 — Handwritten Text Recognition (HTR)](./03_htr_recognition.ipynb)